# 13.3 PageRank — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter13_3_pagerank.ipynb)

책 본문: [13.3 PageRank](https://smhanlab.com/book-ml/kor/ml1/chapter13/3.html)

카라테 클럽 그래프 위에서 13.1절의 무작위 걷기를 "영원히" 돌려,
**각 노드에 머물 확률(PageRank)**을 거듭제곱법으로 직접 계산해봅니다.


In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False
IMG = "/home/smhan/book-ml/kor/src/images"  # 그림 저장 위치(로컬). Colab에서는 이 줄을 수정하세요.


## 1. Zachary's Karate Club

34개 노드, 78개 엣지의 표준 그래프(13.1절과 같은 데이터). 1977년 실제 분열:
**감독파**(node 0, 파랑) vs **관장파**(node 33, 주황). 임베딩과 달리 PageRank는
노드별 스칼라 점수를 주므로, 점수 순서와 실제 파벌 구조가 일치하는지만 봅니다.


In [2]:
KARATE_EDGES = [
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),
    (0,12),(0,13),(0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),
    (1,17),(1,19),(1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),
    (2,28),(2,32),(3,7),(3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),
    (6,16),(8,30),(8,32),(8,33),(9,33),(13,33),(14,32),(14,33),(15,32),
    (15,33),(18,32),(18,33),(19,33),(20,32),(20,33),(22,32),(22,33),
    (23,25),(23,27),(23,29),(23,32),(23,33),(24,25),(24,27),(24,31),
    (25,31),(26,29),(26,33),(27,33),(28,31),(28,33),(29,32),(29,33),
    (30,32),(30,33),(31,32),(31,33),(32,33),
]
N_NODES = 34
FACTION = [0]*34  # 0 = 감독파, 1 = 관장파
for i in [9,14,15,18,20,22,23,24,25,26,27,28,29,30,33]:
    FACTION[i] = 1
DEG = {i: sum(1 for a, b in KARATE_EDGES if a == i or b == i) for i in range(N_NODES)}
print("node 0(감독)의 도:", DEG[0], "  node 33(관장)의 도:", DEG[33])


node 0(감독)의 도: 16   node 33(관장)의 도: 17


## 2. PageRank = 같은 무작위 걷기의 정상분포

13.1절의 무작위 걷기를 "영원히" 돌렸을 때 각 노드에 머물 확률 — 이 질문에
답하는 것이 **PageRank**다. 방정식:

    PR(i) = (1-d)/N + d · Σ_{j→i} PR(j)/outdeg(j),   d = 0.85

재귀식이므로 `1/N`에서 시작해 **거듭제곱법**(power iteration)으로 풀고,
전체 점수의 합이 1임을 확인합니다.


In [3]:
def pagerank(edges, n_nodes, d=0.85, iters=100):
    outgoing = {i: [] for i in range(n_nodes)}
    for a, b in edges:
        outgoing[a].append(b)
    outdeg = {i: max(1, len(outgoing[i])) for i in range(n_nodes)}
    incoming = {i: [] for i in range(n_nodes)}
    for a, b in edges:
        incoming[b].append(a)
    pr = {i: 1.0 / n_nodes for i in range(n_nodes)}
    for _ in range(iters):
        pr = {i: (1 - d) / n_nodes + d * sum(pr[j] / outdeg[j] for j in incoming[i])
              for i in range(n_nodes)}
    return pr

directed_edges = KARATE_EDGES + [(b, a) for a, b in KARATE_EDGES]
scores = pagerank(directed_edges, N_NODES)
top5 = sorted(scores.items(), key=lambda kv: -kv[1])[:5]
print("합 =", round(sum(scores.values()), 10))
for n, v in top5:
    print(f"  node {n:2d} ({'감독파' if FACTION[n]==0 else '관장파'}, 도 {DEG[n]:2d}): PR={v:.5f}")


합 = 1.0
  node 33 (관장파, 도 17): PR=0.10092
  node  0 (감독파, 도 16): PR=0.09700
  node 32 (감독파, 도 12): PR=0.07169
  node  2 (감독파, 도 10): PR=0.05708
  node  1 (감독파, 도  9): PR=0.05288


In [4]:
order = sorted(range(N_NODES), key=lambda i: -scores[i])
plt.figure(figsize=(9, 4))
colors = ["tab:blue" if FACTION[i] == 0 else "tab:orange" for i in order]
plt.bar(range(N_NODES), [scores[i] for i in order], color=colors)
plt.xticks(range(N_NODES), range(N_NODES))
plt.xlabel("Node (sorted by score)"); plt.ylabel("PageRank")
plt.title("Karate club PageRank (d=0.85, symmetrized)")
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(IMG + "/ch14_3_pagerank_karate.svg", bbox_inches="tight")
plt.show()


## 3. 자주 하는 실수: 대칭화 없이 엣지를 방향 그대로 넣기

카라테 그래프는 **무방향**인데 `KARATE_EDGES`를 그대로(무방향 기록 순서대로
방향) 넣으면, 결과가 **엣지 기록 관례**에 따라 달라진다 — node 0(감독,
도 16)이 상위권에서 사라지는 것을 직접 확인한다. (본문 13.2절의
"자주 하는 실수"에서 이 버그가 나오지 않는 이유 — 13.2는 **임베딩**이므로
대칭화 없이도 node 0이 잘 표현됐기 때문 — 과는 별개의 문제다.)


In [5]:
scores_naive = pagerank(KARATE_EDGES, N_NODES)
top5_naive = sorted(scores_naive.items(), key=lambda kv: -kv[1])[:5]
print("대칭화 안 함: 상위 5개 =", [(n, round(v, 4)) for n, v in top5_naive])
ranked = sorted(scores_naive.items(), key=lambda kv: -kv[1])
print("node 0의 순위(1부터):", [i+1 for i, (n, _) in enumerate(ranked) if n == 0][0])
print("node 33의 순위(1부터):", [i+1 for i, (n, _) in enumerate(ranked) if n == 33][0])


대칭화 안 함: 상위 5개 = [(33, 0.0759), (32, 0.028), (31, 0.0135), (16, 0.0125), (6, 0.0079)]
node 0의 순위(1부터): 26
node 33의 순위(1부터): 1


## 4. 수렴 추적: d가 클수록 수렴이 느리다

매 반복의 최대 변화량 max|PR(i)ₜ₊₁ − PR(i)ₜ|를 로그 스케일로 기록한다 —
지수적으로 줄어든다. d=0.85가 d=0.5보다 더 오래 걸린다: 서퍼가 더 긴
"링크 체인"의 영향을 더 오래 유지해야 하기 때문이다.


In [6]:
def pagerank_trace(edges, n_nodes, d, tol=1e-9, max_iters=500):
    outgoing = {i: [] for i in range(n_nodes)}
    for a, b in edges:
        outgoing[a].append(b)
    outdeg = {i: max(1, len(outgoing[i])) for i in range(n_nodes)}
    incoming = {i: [] for i in range(n_nodes)}
    for a, b in edges:
        incoming[b].append(a)
    pr = {i: 1.0 / n_nodes for i in range(n_nodes)}
    trace = []
    for _ in range(max_iters):
        new = {i: (1 - d) / n_nodes + d * sum(pr[j] / outdeg[j] for j in incoming[i])
               for i in range(n_nodes)}
        delta = max(abs(new[i] - pr[i]) for i in range(n_nodes))
        pr = new
        trace.append(delta)
        if delta < tol:
            break
    return trace

tr85 = pagerank_trace(directed_edges, N_NODES, 0.85)
tr50 = pagerank_trace(directed_edges, N_NODES, 0.50)
print(f"d=0.85: {len(tr85)}회 반복에서 1e-9 이하로 수렴")
print(f"d=0.50: {len(tr50)}회 반복에서 1e-9 이하로 수렴")

plt.figure(figsize=(8, 4.5))
plt.semilogy(range(1, len(tr85)+1), tr85, label="d = 0.85")
plt.semilogy(range(1, len(tr50)+1), tr50, label="d = 0.5")
plt.xlabel("Iteration"); plt.ylabel("max |ΔPR| (log scale)")
plt.title("Convergence speed of the PageRank power iteration")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(IMG + "/ch14_3_pagerank_convergence.svg", bbox_inches="tight")
plt.show()


d=0.85: 45회 반복에서 1e-9 이하로 수렴
d=0.50: 19회 반복에서 1e-9 이하로 수렴


## 5. 손으로 푼 값과 대조: 노드 4개

본문 "손으로 한 번"의 예 — A→B, B→C, C→A (3-노드 순환) + D→A, d=0.5.
손 계산: PR(D)=0.125, PR(A)≈0.321, PR(B)≈0.286, PR(C)≈0.268.
코드 결과와 일치하는지 검증한다.


In [7]:
toy_edges = [(0, 1), (1, 2), (2, 0), (3, 0)]   # A,B,C,D (D->A)
toy = pagerank(toy_edges, 4, d=0.5, iters=500)
hand = {0: 0.321, 1: 0.286, 2: 0.268, 3: 0.125}
for i in range(4):
    print(f"PR(node {i}) = {toy[i]:.5f}   (손 계산 {hand[i]:.3f})   일치={abs(toy[i]-hand[i]) < 5e-4}")
print("합 =", round(sum(toy.values()), 10))


PR(node 0) = 0.32143   (손 계산 0.321)   일치=True
PR(node 1) = 0.28571   (손 계산 0.286)   일치=True
PR(node 2) = 0.26786   (손 계산 0.268)   일치=True
PR(node 3) = 0.12500   (손 계산 0.125)   일치=True
합 = 1.0


## 정리

- **PageRank** = 13.1절의 무작위 걷기를 영원히 돌렸을 때의 정상분포 → 노드
  하나당 스칼라 중요도 1개 (13.2절의 Node2Vec이 노드당 **벡터**를 만든 것
 과 대비).
- 거듭제곱법으로 수렴 — "변환을 반복 적용 → 고정점"이라는 구조는 ML2 Chapter 4의
  벨만 방정식·가치 반복과 정확히 같다(바나흐 고정점 정리).
- d가 클수록 서퍼의 "기억"이 길어져 수렴이 느려진다.
- 공통 패턴: **반복 적용 → 특별한 지점(임베딩, 정상분포)으로 수렴**.
